# **Finite Markov Decision Processes for GridWorld Problems**


### **Assignment 2 - Reinforcement Learning - MSc. Computer Science, AUEB [2025-2026]**


> Maria Schoinaki, MSc Student
>
> Department of Informatics, Athens University of Economics and Business
>
> mar.schoinaki@aueb.gr

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def compute_gridworld_value_function(environment_parameters):
    """
    Computes the state-value function v(s) for a gridworld MDP
    by solving the linear system (I - A) v = b.

    environment_parameters = {
        "gamma": float,          # discount factor
        "m": int,                # grid width  (number of columns)
        "n": int,                # grid height (number of rows)
        "random_walk_distribution": [p_up, p_down, p_left, p_right],
        "teleport": array/list of shape (K,4),
        "stay_reward": float,    # reward for a transition that stays inside the grid
        "drop_reward": float,    # reward for an action that tries to move outside the grid
        "special_rewards": array/list of shape (K,3)
    }
    """

    gamma = environment_parameters["gamma"]
    m = environment_parameters["m"]
    n = environment_parameters["n"]

    probs = np.array(environment_parameters["random_walk_distribution"], dtype=float)
    probs = probs / probs.sum()  # normalize, in case probabilities do not sum to 1

    teleports = np.array(environment_parameters.get("teleport", []), dtype=int) #If "teleport" key exists, use its value, οtherwise, use empty list [] (no teleports).
    special_rewards = np.array(environment_parameters.get("special_rewards", []), dtype=float)

    stay_reward = environment_parameters["stay_reward"]
    drop_reward = environment_parameters["drop_reward"]

    # map (x, y) -> index in the vector v
    def idx(x, y): 
        return y * m + x # We store all 2D states (x,y) in a 1D vector v of length N = m*n.

    # look up special reward for state (x,y), if any
    def find_special_reward(x, y):
        for sx, sy, r in special_rewards:
            if int(sx) == x and int(sy) == y:
                return float(r)
        return None

    # if (x,y) is a teleport start, return its target (tx,ty), otherwise None
    def find_teleport_target(x, y):
        for sx, sy, tx, ty in teleports:
            if int(sx) == x and int(sy) == y:
                return int(tx), int(ty)
        return None

    N = m * n
    A = np.zeros((N, N))
    b = np.zeros(N)

    # actions: up, down, left, right
    actions = [
        (0, -1),  # up
        (0,  1),  # down
        (-1, 0),  # left
        (1,  0),  # right
    ]

    # Building the system v = A v + b
    for y in range(n):
        for x in range(m):
            i = idx(x, y)
            total_reward = 0.0

            for a_idx, (dx, dy) in enumerate(actions):
                p_a = probs[a_idx]

                # Check if this state triggers a teleport
                t = find_teleport_target(x, y)
                if t is not None:
                    nx, ny = t
                    j = idx(nx, ny)
                    r_special = find_special_reward(x, y)
                    r = r_special if r_special is not None else stay_reward

                else:
                    nx = x + dx
                    ny = y + dy

                    # Moving outside the grid -> stay in the same state, drop reward
                    if nx < 0 or nx >= m or ny < 0 or ny >= n:
                        j = i
                        r = drop_reward
                    else:
                        j = idx(nx, ny)
                        r_special = find_special_reward(x, y)
                        r = r_special if r_special is not None else stay_reward

                # Contribution to next state's value
                A[i, j] += p_a * gamma

                # Contribution to expected immediate reward
                total_reward += p_a * r

            b[i] = total_reward

    # Solve (I - A) v = b
    M = np.eye(N) - A # np.eye(N) is the N×N identity matrix
    v = np.linalg.solve(M, b)

    # Return the values reshaped as n×m grid
    return v.reshape((n, m))

In [ ]:
env_params = {
    "gamma": 0.9,
    "m": 5,
    "n": 5,
    "random_walk_distribution": [0.25, 0.25, 0.25, 0.25],
    "teleport": [
        [1, 0, 1, 4],  # A -> A', col, row, col, row
        [3, 0, 3, 2],  # B -> B', col, row, col, row
    ],
    "stay_reward": 0.0,
    "drop_reward": -1.0,
    "special_rewards": [
        [1, 0, 10.0],  # A gives +10, col, row, reward
        [3, 0, 5.0],   # B gives +5, col, row, reward
    ],
}

v_grid = compute_gridworld_value_function(env_params)
print(np.round(v_grid, 1))


[[ 3.3  8.8  4.4  5.3  1.5]
 [ 1.5  3.   2.3  1.9  0.5]
 [ 0.1  0.7  0.7  0.4 -0.4]
 [-1.  -0.4 -0.4 -0.6 -1.2]
 [-1.9 -1.3 -1.2 -1.4 -2. ]]
